##### Testing Sampling Against Papers
This will be used to evaluate the performance of the package against reference peer reviewed papers.

!["alt"](../images/franks2014Table3.png "Franks2014 p848 Table3")

In [ ]:
import networkx as nx
from Influence_Modelling import XsvNetwork, NetworkMetrics, Sampling

import matplotlib.pyplot as plt
import pandas as pd
import random

In [ ]:
enron = XsvNetwork.XsvNetwork("", "", "enron")
enron_nx = enron.get_nx()

airports = XsvNetwork.XsvNetwork("", "", "airports")
airports_nx = airports.get_nx()

twitter = XsvNetwork.XsvNetwork("", "", "twitter")
twitter_nx = twitter.get_nx()

p2p = XsvNetwork.XsvNetwork("", "", "p2p")
p2p_nx = p2p.get_nx()

The below function will be used to generate the data that is held in the rows of the above table.

In [ ]:
def multiple_runs(
    G: nx.DiGraph, label: str, sampling_method, trials: int = 10, nodes: int = 1000
):
    results = pd.DataFrame()
    for _ in range(trials):
        sample: nx.DiGraph = sampling_method(G, nodes)
        results = pd.concat(
            [
                results,
                pd.DataFrame(
                    data={
                        "nodes": [NetworkMetrics.s_1_number_nodes(sample)],
                        "mean_degree": [NetworkMetrics.s_3_mean_degree(sample)],
                        "cc": [NetworkMetrics.s_4_clustering_coefficient(sample)],
                        #'centralization': [NetworkMetrics.s_6_centralization(input_graph)]
                    }
                ),
            ]
        )
    df = pd.DataFrame(
        results,
        columns=["nodes", "edges", "mean_degree", "cc"],  # , "centralization"]
    )
    return pd.DataFrame(
        data={
            "method": [label],
            "nodes": [df["nodes"].to_numpy().mean()],
            "nodes_std": [df["nodes"].to_numpy().std()],
            "edges": [df["edges"].to_numpy().mean()],
            "edges_std": [df["edges"].to_numpy().std()],
            "mean_degree": [df["mean_degree"].to_numpy().mean()],
            "mean_degree_std": [df["mean_degree"].to_numpy().std()],
            "cc": [df["cc"].to_numpy().mean()],
            "cc_std": [df["cc"].to_numpy().std()],
            #'centralization': [df['centralization'].to_numpy().mean()],
            #'centralization_std': [df['centralization'].to_numpy().std()],
        }
    )


def generate_table_3_2014_learning():
    trials = 15
    nodes = 500
    datasets = [(enron_nx, "enron"), (p2p_nx, "p2p")]  # ... airports dataset etc.. 
    methods = [
        (Sampling.mhrw_sample, "MHRW"),
        (Sampling.mhda_sample, "MHDA"),
    ]  # , (Sampling.snowball_sample, "SNS")] etc...
    results = []
    print("Generating Table 3 Franks 2014 Learning Paper")
    for data, label in datasets:
        print(f"Started {label} Table -----------")
        method_runs = []
        for method, method_name in methods:
            print(f"Started {label}-{method_name} Sampling")
            method_runs.append(multiple_runs(data, method_name, method, trials, nodes))
            print(f"Completed {label}-{method_name} Sampling")
        table: pd.DataFrame = pd.concat(method_runs)
        results.append(table)
        print(f"Completed {label} Table -----------")
    return results


enron_sampled = generate_table_3_2014_learning()

Generating Table 3 Franks 2014 Learning Paper
Started enron Table -----------
Started enron-MHRW Sampling
Completed enron-MHRW Sampling
Started enron-MHDA Sampling
Completed enron-MHDA Sampling
Completed enron Table -----------
Started p2p Table -----------
Started p2p-MHRW Sampling
Completed p2p-MHRW Sampling
Started p2p-MHDA Sampling
Completed p2p-MHDA Sampling
Completed p2p Table -----------


In [ ]:
enron_sampled[0]
enron_sampled[1]


,method,nodes,nodes_std,edges,edges_std,mean_degree,mean_degree_std,cc,cc_std
0,MHRW,10.0,0.0,12.6,2.154066,2.52,0.430813,0.320762,0.189595
0,MHDA,10.0,0.0,7.0,3.577709,1.40,0.715542,0.077619,0.155238


In [ ]:
def mhrw_sample(G: nx.DiGraph, n: int) -> nx.DiGraph:
    """
    Metropolis-Hastings Random Walk. A random walk sample with a bias
    towards high degree locations aiming to preserve them inside the sample.

    Based on pseudocode from `Practical Recommendations on Crawling social networks (page 5)`

    :param nx.Digraph G: The graph to be sampled
    :param int n: Number of nodes to be selected in our sample.
    :return: Subgraph of G following the algorithm
    """
    if n >= len(list(G.nodes())):
        return G
    subgraph_nodes = set()
    v = Sampling.random_sample([v for v in G.nodes()], 1)[0]

    while len(subgraph_nodes) < n:
        neighbours = [node for node in G.adj[v]]

        # remove already contained nodes, TODO use map
        for n in neighbours:
            if n in subgraph_nodes:
                neighbours.remove(n)

        if len(neighbours) == 0:
            neighbours = [u for u in G.nodes() if u not in subgraph_nodes]

        random.shuffle(neighbours)
        w = neighbours[0]
        p = random.random()
        if p <= (G.degree(v)) / (G.degree(w)):
            subgraph_nodes.add(v)
            v = w
    return G.subgraph(subgraph_nodes)
